# CMA-ES with restarts — a derivative-free global search, measured against the gradient

`h_train` is 500 independent 10-dimensional optimisation problems, and what decides the answer
is not the optimiser but which **basin** the run starts in. Notebook 03 attacks that by brute
sampling: cover the box with 65k Sobol points and Adam-refine the survivors. This notebook
attacks it with the standard tool for rugged non-convex continuous problems in exactly this
dimension — **CMA-ES with restarts** (IPOP-CMA-ES, Auger & Hansen 2005), the long-standing
reference method on the BBOB benchmark suite.

**Why this is not obviously a good idea, stated up front.** CMA-ES is *derivative-free*. Its
whole reason to exist is objectives whose gradient you cannot get — and here the gradient is
free and exact, because `QAOA.py` is written in torch. Spending samples to infer a descent
direction you could have had for a third of the price is a real handicap.

Whether it pays for itself turns out to depend on the budget. A small CPU probe, matched
evaluations, best arm of each kind:

| eval-equivalents per instance | best Adam multistart | best CMA-ES |
|---|---|---|
| 11,500 | **0.233** | 0.217 |
| 24,000 | 0.242 | **0.274** |

Below the crossover CMA-ES is still paying off the cost of learning a covariance and loses; above
it, the learned metric starts reaching basins that gradient descent from random starts does not.
Those are 6-8 instances on a CPU and are indicative only — §7 re-measures the comparison properly,
on the GPU, at the budget actually being used.

Why it can win at all, despite the handicap: the two methods are not competing at the same job.

| | Adam | CMA-ES |
|---|---|---|
| moving **inside** a basin | very fast, exact gradient | slow — it has to learn the metric first |
| choosing **which** basin | cannot; it follows the local slope | step size expands and contracts, so it can leave one |
| ill-conditioned valleys | needs the right `lr` | adapts the full covariance, becomes scale-free |

That splits the labour: **CMA-ES selects basins, Adam finishes them.** The search here is
therefore a hybrid, and — as in §4 of notebook 03 — the arms are *measured* against each other
at a matched budget in §7 rather than assumed.

**What "with restarts" means.** Restarts are the only part of CMA-ES that does global search,
so this notebook does them at two scales:

- **slot recycling** — every run is watched with Hansen's standard termination criteria
  (`tolfun`, `tolx`, condition number, stagnation, step-size divergence). A converged run is
  worthless from then on, so its GPU slot is reinitialised in place with a fresh mean and the GPU
  never idles. A configurable share of those reseeds draws from the **elite pool** — angles that
  already won on *other* instances — which is the parameter-concentration effect notebook 03
  exploits in §5.

  *How* aggressively to recycle is itself a live question, and the intuitive answer is wrong. The
  same probe found that tightening the triggers to fire every ~25 generations **cost** 10%
  (0.247 against 0.274), because runs were being killed while still improving. The defaults below
  are therefore Hansen's, deliberately conservative, and §7 measures the aggressive setting as its
  own arm rather than leaving it to taste.
- **IPOP waves** — each wave doubles the population size `lambda`. Small populations converge
  fast and locally, large ones behave almost like a global search. §3 shows the effect starkly
  on Rastrigin: `lambda=10` never gets below f=7, `lambda=200` finds the exact global optimum.

Everything is batched: `runs x 500` independent CMA-ES instances advance in lockstep inside the
same CUDA kernels, so the covariance algebra costs nothing next to the simulator calls.

## 0. Setup

In [ ]:
import os, sys, glob, time, math, csv, json
import numpy as np
import torch
from torch.quasirandom import SobolEngine
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
else:
    print("  WARNING: no GPU. Set QUICK = True below or this will take hours.")

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)


def find_file(name):
    """Locate an organiser file across Kaggle input, Colab, or a local checkout."""
    for root in ["/kaggle/input", "/kaggle/working", "data/raw", "../data/raw", ".", "..", "/content"]:
        if os.path.isdir(root):
            hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
            if hits:
                return hits[0]
    raise FileNotFoundError(
        f"Could not find {name}. Attach the competition files as a Kaggle dataset "
        f"(J.npy, h_train.npy, QAOA.py) or put them in ./data/raw/.")


QAOA_PATH = find_file("QAOA.py")
sys.path.insert(0, os.path.dirname(os.path.abspath(QAOA_PATH)))
from QAOA import QAOA, P as P_DEPTH, N_QUBITS

J = np.load(find_file("J.npy")).astype(np.float64)
h_train = np.load(find_file("h_train.npy")).astype(np.float64)
qaoa = QAOA(torch.tensor(J, dtype=torch.float32), device=DEVICE)

D_ANG = 2 * P_DEPTH          # 10 free angles: 5 gamma + 5 beta
GAMMA_MAX = math.pi          # gamma box; beta spans exactly one period, (-pi/2, pi/2)
print(f"\nJ {J.shape} | h_train {h_train.shape} | p={P_DEPTH} | n={N_QUBITS} | d={D_ANG}")

## 1. Configuration

The budget is stated once, in **forward-evaluation-equivalents per instance**, so it is directly
comparable with notebook 03 (a gradient step costs about 3 forward evaluations — forward, backward,
optimiser). Every stage below is a slice of that one number.

- **`cma_evals`** — the search budget, split evenly across IPOP waves. The main quality knob.
- **`runs` x `lam`** — how the budget is spent per generation: `runs` independent CMA-ES
  instances *per problem instance*, each drawing `lam` samples. More runs = more basins visited
  in parallel; larger `lam` = each run is individually more global. Wave `w` uses `lam * 2**w`
  and correspondingly fewer generations.
- **`keep` / `polish`** — the Adam endgame. CMA-ES gets the basin to a few decimal places;
  gradient descent takes it the rest of the way for a fraction of the cost.
- **`p_elite`** — share of restarts reseeded from angles that won on other instances.
- **`*_rows`** — GPU memory only. Forward evaluation holds ~3 live `(rows, 4096)` complex
  tensors (16384 rows ~ 1.6 GiB); Adam holds ~65 of them (2048 rows ~ 4.4 GiB). Halve on OOM.

In [ ]:
CFG = dict(
    # --- search budget -------------------------------------------------------
    cma_evals = 90_000,   # forward evals per instance spent inside CMA-ES
    waves     = 2,        # IPOP waves; wave w uses lam * 2**w
    runs      = 8,        # parallel CMA-ES runs per problem instance
    lam       = 16,       # population size in wave 0
    sigma0    = 0.40,     # initial step size (angle box half-width is ~pi)
    # --- restart triggers: Hansen's defaults. Tightening these lost 10% in the probe
    #     (see the header) -- runs get killed while still improving. §7 arm E re-tests it.
    tolfun    = 1e-11,    # best-f range over `hist_len` generations
    tolx      = 1e-11,    # sigma * spread of C collapsed
    stag_gens = 100,      # generations without improving this run's own best
    hist_len  = 20,
    p_elite   = 0.25,     # share of restarts reseeded from other instances' winners
    # --- seeding mix for wave starts (uniform, ramp, screened) ---------------
    init      = (0.50, 0.25, 0.25),
    n_screen  = 4096,     # Sobol points scored to place the 'screened' means
    # --- Adam endgame --------------------------------------------------------
    keep      = 16,       # candidates handed to the polish
    polish    = 250,      # Adam steps on them
    polish_lr = 0.05,
    # --- memory / speed ------------------------------------------------------
    eval_rows = 16384,
    adam_rows = 2048,
    eig_every = None,     # generations between eigendecompositions of C; None = Hansen's rule
)                         #   (here: every generation). Raise to 3-5 if batched eigh is slow.
QUICK = False             # True -> tiny run to check the notebook end to end
if QUICK:
    CFG.update(cma_evals=4000, runs=4, lam=8, keep=4, polish=40, n_screen=256, stag_gens=12)
    h_train = h_train[:24]        # smoke test only -- the printed scores are not comparable
    print(f"QUICK: h_train truncated to {len(h_train)} instances\n")


def budget(cfg):
    """Forward-eval-equivalents per instance, by stage. 1 Adam step = 3 forward evals."""
    per_wave = cfg["cma_evals"] // cfg["waves"]
    gens = [max(1, per_wave // (cfg["runs"] * cfg["lam"] * 2 ** w)) for w in range(cfg["waves"])]
    cma = sum(g * cfg["runs"] * cfg["lam"] * 2 ** w for w, g in enumerate(gens))
    return gens, dict(screen=cfg["n_screen"] * (cfg["init"][2] > 0) * cfg["waves"],
                      cma=cma, polish=cfg["keep"] * cfg["polish"] * 3)


GENS, WORK = budget(CFG)
print(json.dumps(CFG, indent=2))
print(f"\ngenerations per wave: {GENS}   (lam = {[CFG['lam'] * 2**w for w in range(CFG['waves'])]})")
print(f"CMA-ES runs held on the GPU at once: {CFG['runs']} x 500 = {CFG['runs']*500}")
print("\nforward-eval-equivalents per instance:")
for k, v in WORK.items():
    print(f"  {k:7s}: {v:9,d}  ({v/sum(WORK.values())*100:4.1f}%)")
print(f"  {'total':7s}: {sum(WORK.values()):9,d}     (notebook 03 spends ~152,000)")

## 2. Batched CMA-ES

One class, following Hansen's *CMA Evolution Strategy: A Tutorial* (2016). Every tensor carries a
leading `S` axis: `S` runs that never interact, batched only so a single GPU kernel advances all
of them at once. With `d = 10` the covariance work is `S` copies of a 10x10 eigendecomposition,
which is noise next to `S * lam` simulator calls.

The loop is the standard one:

1. **sample** `x = m + sigma * B diag(D) z`, `z ~ N(0, I)` — `B diag(D)` is the square root of the
   covariance `C`, so the sample cloud has `C`'s shape;
2. **select** the best `mu = lam/2` and move `m` to their weighted mean;
3. **adapt** two evolution paths — an isotropic one (`ps`) that compares the realised step length
   against the expected one and drives `sigma`, and an anisotropic one (`pc`) that feeds the rank-1
   covariance update;
4. **update** `C` from a rank-1 term (the path) plus a rank-`mu` term (this generation's spread).

`restart()` is what makes it a *global* method, and `stopped()` decides when: the run has collapsed
(`tolx`), stopped improving (`tolfun`, `stagnation`), gone numerically degenerate (`cond`), or
diverged (`sigma`). Reasons are recorded exclusively, in that priority order, so they can be
plotted as a breakdown in §8.

In [ ]:
STOP_REASONS = ("tolx", "tolfun", "cond", "stagnation", "sigma")


class BatchCMA:
    """S independent (mu/mu_w, lambda)-CMA-ES runs advanced in lockstep on the GPU."""

    def __init__(self, m0, sigma0, lam, resample, seed=0, eig_every=None,
                 tolx=1e-11, tolfun=1e-12, cond_max=1e13, stag_gens=120,
                 hist_len=20, sigma_max=1e3):
        self.S, self.d = m0.shape
        d, S = self.d, self.S
        self.dev = m0.device
        self.lam, self.mu = int(lam), int(lam) // 2
        self.resample = resample
        self.g = torch.Generator(device=self.dev).manual_seed(seed)

        # --- selection weights: log-decreasing over the best mu of lam -------------
        w = torch.log(torch.tensor(self.mu + 0.5, device=self.dev)) - \
            torch.arange(1, self.mu + 1, device=self.dev, dtype=torch.float32).log()
        self.w = w / w.sum()
        self.mueff = float(1.0 / (self.w ** 2).sum())      # variance-effective selection mass

        # --- adaptation constants: Hansen's defaults, functions of d and mueff -----
        me = self.mueff
        self.cc = (4 + me / d) / (d + 4 + 2 * me / d)          # pc learning rate
        self.cs = (me + 2) / (d + me + 5)                      # ps learning rate
        self.c1 = 2 / ((d + 1.3) ** 2 + me)                    # rank-1 rate
        self.cmu = min(1 - self.c1, 2 * (me - 2 + 1 / me) / ((d + 2) ** 2 + me))
        self.damps = 1 + 2 * max(0.0, math.sqrt((me - 1) / (d + 1)) - 1) + self.cs
        self.chiN = math.sqrt(d) * (1 - 1 / (4 * d) + 1 / (21 * d * d))   # E||N(0,I)||
        # lazy eigendecomposition: refreshing B,D every generation is wasted work
        self.eig_every = eig_every or max(1, int(1 / ((self.c1 + self.cmu) * d * 10)))

        self.tolx, self.tolfun, self.cond_max = tolx, tolfun, cond_max
        self.stag_gens, self.hist_len, self.sigma_max = stag_gens, hist_len, sigma_max
        self.sigma0 = float(sigma0)
        self.stop_counts = dict.fromkeys(STOP_REASONS, 0)
        self._init_state(torch.ones(S, dtype=torch.bool, device=self.dev), m0)
        self.total_gens = 0

    # ------------------------------------------------------------------ state ----
    def _init_state(self, mask, m_new=None):
        """(Re)initialise the runs selected by `mask` — used at start and on restart."""
        S, d, dev = self.S, self.d, self.dev
        if not hasattr(self, "m"):
            z = lambda *s: torch.zeros(*s, device=dev)
            self.m, self.ps, self.pc, self.xbest = z(S, d), z(S, d), z(S, d), z(S, d)
            self.C, self.B, self.D = z(S, d, d), z(S, d, d), z(S, d)
            self.sigma, self.gen, self.last_improve = z(S), z(S), z(S)
            self.restarts = torch.zeros(S, dtype=torch.long, device=dev)
            self.fbest = torch.full((S,), float("inf"), device=dev)
            self.fhist = torch.full((S, self.hist_len), float("nan"), device=dev)
        k = int(mask.sum())
        if k == 0:
            return
        eye = torch.eye(d, device=dev).expand(k, d, d)
        self.m[mask] = (self.resample(k) if m_new is None else m_new).to(dev, torch.float32)
        self.sigma[mask] = self.sigma0
        self.C[mask], self.B[mask], self.D[mask] = eye, eye, 1.0
        self.ps[mask] = self.pc[mask] = 0.0
        self.gen[mask] = self.last_improve[mask] = 0.0
        self.fhist[mask] = float("nan")
        self.fbest[mask] = float("inf")       # run-local; the global best is banked by the caller

    # ------------------------------------------------------------------- ask -----
    def ask(self):
        """Sample one population per run: x = m + sigma * B diag(D) z, z ~ N(0, I)."""
        S, d, lam = self.S, self.d, self.lam
        z = torch.randn(S, lam, d, device=self.dev, generator=self.g)
        BD = self.B * self.D.unsqueeze(-2)                 # columns of B scaled by D
        y = torch.einsum("sij,slj->sli", BD, z)
        self._z, self._y = z, y
        return self.m.unsqueeze(1) + self.sigma.view(S, 1, 1) * y

    # ------------------------------------------------------------------ tell -----
    def tell(self, f):
        """Update mean, paths, covariance and step size from f: (S, lam), minimised."""
        S, d, mu = self.S, self.d, self.mu
        f = torch.nan_to_num(f, nan=float("inf"))
        pick = f.argsort(dim=1)[:, :mu].unsqueeze(-1).expand(S, mu, d)
        ys, zs = self._y.gather(1, pick), self._z.gather(1, pick)
        wv = self.w.view(1, mu, 1)
        yw, zw = (wv * ys).sum(1), (wv * zs).sum(1)

        self.m = self.m + self.sigma.unsqueeze(1) * yw
        self.gen = self.gen + 1

        # isotropic path: realised vs expected step length -> step-size control
        self.ps = (1 - self.cs) * self.ps + math.sqrt(self.cs * (2 - self.cs) * self.mueff) \
            * torch.einsum("sij,sj->si", self.B, zw)
        nps = self.ps.norm(dim=1)
        denom = (1 - (1 - self.cs) ** (2 * self.gen)).clamp_min(1e-12).sqrt()
        hsig = ((nps / denom / self.chiN) < (1.4 + 2 / (d + 1))).float()

        # anisotropic path -> rank-1 term; this generation's spread -> rank-mu term
        self.pc = (1 - self.cc) * self.pc + \
            (hsig * math.sqrt(self.cc * (2 - self.cc) * self.mueff)).unsqueeze(1) * yw
        dh = (1 - hsig) * self.cc * (2 - self.cc)          # variance lost when hsig == 0
        rank1 = self.pc.unsqueeze(2) * self.pc.unsqueeze(1)
        rankmu = torch.einsum("m,smi,smj->sij", self.w, ys, ys)
        self.C = (1 - self.c1 - self.cmu + self.c1 * dh).view(S, 1, 1) * self.C \
            + self.c1 * rank1 + self.cmu * rankmu

        self.sigma = (self.sigma * torch.exp((self.cs / self.damps) * (nps / self.chiN - 1))) \
            .clamp(1e-14, self.sigma_max)

        self.total_gens += 1
        if self.total_gens % self.eig_every == 0:
            self._eig()

        fmin = f.min(dim=1).values
        improved = fmin < self.fbest
        self.fbest = torch.where(improved, fmin, self.fbest)
        self.last_improve = torch.where(improved, torch.zeros_like(self.last_improve),
                                        self.last_improve + 1)
        self.fhist = torch.roll(self.fhist, -1, dims=1)
        self.fhist[:, -1] = fmin

    def _eig(self):
        self.C = 0.5 * (self.C + self.C.transpose(1, 2))       # kill drift out of symmetry
        bad = ~torch.isfinite(self.C).flatten(1).all(1)
        if bad.any():
            self.C[bad] = torch.eye(self.d, device=self.dev)
        ev, self.B = torch.linalg.eigh(self.C)
        self.D = ev.clamp_min(1e-20).sqrt()

    # --------------------------------------------------------------- restarts ----
    def stopped(self):
        """Per-run stopping mask + exclusive reason code (index into STOP_REASONS, -1 = running)."""
        dC = torch.diagonal(self.C, dim1=1, dim2=2).clamp_min(0).sqrt()
        full = ~self.fhist.isnan().any(dim=1)
        tests = [
            self.sigma.unsqueeze(1).mul(torch.maximum(dC, self.pc.abs())).max(1).values < self.tolx,
            full & ((self.fhist.max(1).values - self.fhist.min(1).values) < self.tolfun),
            (self.D.max(1).values / self.D.min(1).values.clamp_min(1e-20)) > self.cond_max,
            self.last_improve > self.stag_gens,
            (self.sigma >= self.sigma_max) | ~torch.isfinite(self.sigma),
        ]
        reason = torch.full((self.S,), -1, dtype=torch.long, device=self.dev)
        for i in range(len(tests) - 1, -1, -1):                # earlier tests win
            reason = torch.where(tests[i], torch.full_like(reason, i), reason)
        for i, name in enumerate(STOP_REASONS):
            self.stop_counts[name] += int((reason == i).sum())
        return reason >= 0, reason

    def restart(self, mask, m_new=None):
        if int(mask.sum()) == 0:
            return 0
        self.restarts[mask] += 1
        self._init_state(mask, m_new)
        return int(mask.sum())

## 3. Does the implementation actually work?

A hand-written CMA-ES that is subtly wrong still *looks* like it is optimising — it just quietly
behaves like a random search with a step size. So before pointing it at QAOA, run it on the
benchmarks the algorithm is defined by. These are the standard checks:

| function | what it proves |
|---|---|
| **sphere** | the basic (mu/mu_w, lambda) loop and step-size control converge |
| **ellipsoid** (cond `1e6`) | the *covariance* adaptation works — without it this stalls |
| **Rosenbrock** | the run can follow a bent, ill-conditioned valley; the single hardest test of a CMA implementation |
| **Rastrigin** | multimodal. Fails with a small population, succeeds with a large one — this is exactly the effect the IPOP waves in §1 are buying |

All four in 10 dimensions, the same `d` as the angle problem.

In [ ]:
def _bench(fn, S, lam, gens, spread=3.0, sigma0=0.5, restarts=False, seed=0, **kw):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    rs = lambda k: (torch.rand(k, 10, device=DEVICE, generator=g) * 2 - 1) * spread
    es = BatchCMA(rs(S), sigma0, lam, rs, seed=seed, **kw)
    best = torch.full((S,), float("inf"), device=DEVICE)
    for _ in range(gens):
        x = es.ask()
        f = fn(x)
        es.tell(f)
        best = torch.minimum(best, f.min(dim=1).values)
        if restarts:
            es.restart(es.stopped()[0])
    return best


sphere = lambda x: (x ** 2).sum(-1)
rosen  = lambda x: (100 * (x[..., 1:] - x[..., :-1] ** 2) ** 2 + (1 - x[..., :-1]) ** 2).sum(-1)
_w = torch.tensor([1e6 ** (i / 9) for i in range(10)], device=DEVICE)
ellip  = lambda x: (_w * x ** 2).sum(-1)
rast   = lambda x: 10 * 10 + (x ** 2 - 10 * torch.cos(2 * math.pi * x)).sum(-1)

t0 = time.time()
G = (60, 60, 60) if QUICK else (300, 800, 1000)     # QUICK: too few gens to converge, asserts off
r_sph = _bench(sphere, 8, 10, G[0])
r_ell = _bench(ellip,  8, 10, G[1])
r_ros = _bench(rosen,  8, 10, G[2])
rg = 60 if QUICK else 400
r_small = _bench(rast, 16, 10,  rg, restarts=True, seed=1, tolfun=1e-9, stag_gens=40)
r_large = _bench(rast, 16, 200, rg, restarts=True, seed=1, tolfun=1e-9, stag_gens=40)

print(f"{'function':22s} {'best':>11s} {'median':>11s}   expected")
print(f"{'sphere':22s} {r_sph.min():11.2e} {r_sph.median():11.2e}   -> 0")
print(f"{'ellipsoid cond=1e6':22s} {r_ell.min():11.2e} {r_ell.median():11.2e}   -> 0")
print(f"{'rosenbrock':22s} {r_ros.min():11.2e} {r_ros.median():11.2e}   -> 0 (some runs trap at ~3.99)")
print(f"{'rastrigin lam=10':22s} {r_small.min():11.2e} {r_small.median():11.2e}   -> stuck, small population")
print(f"{'rastrigin lam=200':22s} {r_large.min():11.2e} {r_large.median():11.2e}   -> global optimum found")
print(f"\n{time.time()-t0:.0f}s")

if not QUICK:
    assert r_sph.median() < 1e-12,  "sphere: the base loop is broken"
    assert r_ell.median() < 1e-15,  "ellipsoid: covariance adaptation is broken"
    assert r_ros.median() < 1e-6,   "rosenbrock: path/covariance coupling is broken"
    assert r_large.min() < r_small.min(), "population size has no effect -> restarts are broken"
    print("all checks passed")

## 4. The objective, and where the runs start

`p_ground` is the thing being maximised; CMA-ES minimises, so the search works on `-P`. Both
helpers below chunk over rows purely to bound GPU memory.

Wave means come from three families, mixed by `CFG['init']` — the same idea as notebook 03's
candidate pool, but placing ~12 *distributions* per instance rather than 65k points:

1. **uniform** — a random point in the angle box; the neutral baseline and what plain IPOP does.
2. **ramp** — a discretised adiabatic schedule (`gamma` rising, `beta` falling) with jitter. This
   family lands in a good basin far more often than chance.
3. **screened** — the best of a small Sobol sample, scored per instance. A cheap way to spend a
   few thousand evaluations on *placing* the means instead of on the search itself.

§8 reports which family actually produced the winners, so the mix is checkable rather than
assumed.

In [ ]:
@torch.no_grad()
def eval_p(h_rows, ang, chunk):
    """P(ground) for aligned (h_rows, ang) pairs, chunked to bound memory. (M,12),(M,10)->(M,)."""
    out = torch.empty(len(ang), device=DEVICE)
    for s in range(0, len(ang), chunk):
        e = s + chunk
        out[s:e] = qaoa.p_ground(h_rows[s:e], ang[s:e, :P_DEPTH], ang[s:e, P_DEPTH:])
    return out


def adam_refine(h, cand, steps, lr, chunk_rows, keep):
    """Adam-ascend every candidate of every instance; keep the best `keep`, best-first.

    Each candidate is compared against where it started and the better of the two survives, so
    this stage can never score worse than its input. cand: (N,K,10) -> ((N,keep,10), (N,keep)).
    """
    n, k, d = cand.shape
    n_h = max(1, chunk_rows // k)
    out_c = torch.empty(n, keep, d, device=DEVICE)
    out_p = torch.empty(n, keep, device=DEVICE)
    for s in range(0, n, n_h):
        hc = h[s:s + n_h]
        c = len(hc)
        hb = hc.repeat_interleave(k, 0)
        a0 = cand[s:s + c].reshape(c * k, d).clone()
        p0 = eval_p(hb, a0, chunk_rows)
        a = a0.clone().requires_grad_(True)
        opt = torch.optim.Adam([a], lr=lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr / 25)
        for _ in range(steps):
            opt.zero_grad()
            (-qaoa.p_ground(hb, a[:, :P_DEPTH], a[:, P_DEPTH:]).sum()).backward()
            opt.step()
            sch.step()
        p1 = eval_p(hb, a.detach(), chunk_rows)
        better = (p1 >= p0).unsqueeze(1)
        ab = torch.where(better, a.detach(), a0).view(c, k, d)
        v, i = torch.maximum(p1, p0).view(c, k).topk(keep, dim=1)
        out_p[s:s + c] = v
        out_c[s:s + c] = ab.gather(1, i.unsqueeze(-1).expand(c, keep, d))
    return out_c, out_p


# ------------------------------------------------------------- seeding families --
FAMILIES = ("uniform", "ramp", "screened", "restart-rand", "restart-elite")


def draw_uniform(k, gen):
    g = (torch.rand(k, P_DEPTH, device=DEVICE, generator=gen) * 2 - 1) * GAMMA_MAX
    b = (torch.rand(k, P_DEPTH, device=DEVICE, generator=gen) * 2 - 1) * (math.pi / 2)
    return torch.cat([g, b], 1)


def draw_ramp(k, gen):
    """Discretised adiabatic schedules with jitter: gamma ramps up, beta ramps down."""
    dt = torch.rand(k, 1, device=DEVICE, generator=gen) * 1.2 + 0.2
    l = torch.arange(P_DEPTH, device=DEVICE).view(1, -1)
    a = torch.cat([(l + 1) / P_DEPTH * dt, (1 - l / P_DEPTH) * dt], 1)
    return a + 0.15 * torch.randn(a.shape, device=DEVICE, generator=gen)


@torch.no_grad()
def draw_screened(h, k, n_pool, gen, seed, chunk):
    """Best `k` of a small scrambled-Sobol sample, per instance. h: (N,12) -> (N,k,10)."""
    u = SobolEngine(D_ANG, scramble=True, seed=seed).draw(n_pool).to(DEVICE)
    pool = torch.cat([(u[:, :P_DEPTH] * 2 - 1) * GAMMA_MAX,
                      (u[:, P_DEPTH:] * 2 - 1) * (math.pi / 2)], 1)
    n = len(h)
    out = torch.empty(n, k, D_ANG, device=DEVICE)
    per = max(1, chunk // n_pool)
    for s in range(0, n, per):
        c = len(h[s:s + per])
        p = eval_p(h[s:s + per].repeat_interleave(n_pool, 0), pool.repeat(c, 1), chunk).view(c, n_pool)
        out[s:s + c] = pool[p.topk(k, dim=1).indices]
    return out


def seed_means(h, cfg, gen, seed):
    """Initial means for every (instance, run) slot + the family label of each. -> (N*runs,10),(N*runs,)"""
    n, runs = len(h), cfg["runs"]
    fu, fr, fs = cfg["init"]
    n_s = int(round(runs * fs / (fu + fr + fs)))
    n_r = int(round(runs * fr / (fu + fr + fs)))
    n_u = runs - n_s - n_r
    parts, fams = [], []
    if n_u:
        parts.append(draw_uniform(n * n_u, gen).view(n, n_u, D_ANG)); fams += [0] * n_u
    if n_r:
        parts.append(draw_ramp(n * n_r, gen).view(n, n_r, D_ANG)); fams += [1] * n_r
    if n_s:
        parts.append(draw_screened(h, n_s, cfg["n_screen"], gen, seed, cfg["eval_rows"])); fams += [2] * n_s
    m = torch.cat(parts, 1)                                  # (n, runs, 10)
    fam = torch.tensor(fams, device=DEVICE).repeat(n)
    return m.reshape(n * runs, D_ANG), fam

## 5. The search loop

The part that makes this a global method. Per generation:

1. every slot samples `lam` angle vectors; all `N * runs * lam` of them are scored in one batch;
2. the results go back into CMA-ES;
3. each instance's running **top-`keep`** is updated — every evaluation ever made is a candidate,
   so nothing found is lost when a run is later thrown away;
4. converged runs are detected and **immediately restarted in place**, a `p_elite` share of them
   reseeded from angles that won on *other* instances.

Because slots are recycled rather than retired, a wave of `gens` generations contains far more
than `runs` distinct CMA-ES runs — §8 reports how many. `history` records one row per generation
so the run can be read afterwards instead of guessed at.

In [ ]:
def cma_search(h_np, cfg, seed=0, log=True):
    """IPOP-wave CMA-ES with slot recycling. -> (top_x (N,keep,10), top_p (N,keep), hist, meta)."""
    h = torch.as_tensor(h_np, dtype=torch.float32, device=DEVICE)
    n, runs, keep = len(h), cfg["runs"], cfg["keep"]
    S = n * runs
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    gens_per_wave, _ = budget(cfg)

    top_x = torch.zeros(n, keep, D_ANG, device=DEVICE)
    top_p = torch.zeros(n, keep, device=DEVICE)
    best_fam = torch.full((n,), -1, dtype=torch.long, device=DEVICE)
    hist = {k: [] for k in ("wave", "gen", "evals", "best_mean", "best_med", "pop_mean",
                            "sigma_med", "sigma_lo", "sigma_hi", "cond_med", "restarts", "t")}
    meta = dict(restart_src={f: 0 for f in FAMILIES[3:]}, stop_counts=dict.fromkeys(STOP_REASONS, 0),
                total_runs=0, waves=[])
    t0, ev_cum, n_restart, gstep, have_elite = time.time(), 0, 0, 0, False

    for w, gens in enumerate(gens_per_wave):
        lam = cfg["lam"] * 2 ** w
        m0, fam = seed_means(h, cfg, gen, seed + w)
        es = BatchCMA(m0, cfg["sigma0"], lam, lambda k: draw_uniform(k, gen), seed=seed * 97 + w,
                      tolx=cfg["tolx"], tolfun=cfg["tolfun"], stag_gens=cfg["stag_gens"],
                      hist_len=cfg["hist_len"], eig_every=cfg.get("eig_every"))
        meta["total_runs"] += S
        h_rows = h.repeat_interleave(runs, 0).repeat_interleave(lam, 0)     # (S*lam, 12)

        for g in range(gens):
            x = es.ask()                                                     # (S, lam, 10)
            p = eval_p(h_rows, x.reshape(-1, D_ANG), cfg["eval_rows"]).view(S, lam)
            es.tell(-p)
            ev_cum += runs * lam

            # --- bank: every sample is a candidate, merged into the per-instance top-K
            xa = torch.cat([top_x, x.view(n, runs * lam, D_ANG)], 1)
            pa = torch.cat([top_p, p.view(n, runs * lam)], 1)
            v, i = pa.topk(keep, dim=1)
            top_p, top_x = v, xa.gather(1, i.unsqueeze(-1).expand(n, keep, D_ANG))
            have_elite = True                    # every instance now has a scored best

            # --- attribute each instance's current winner to a seeding family
            gv, gi = p.view(n, runs * lam).max(1)
            imp = gv >= top_p[:, 0]
            if imp.any():
                best_fam[imp] = fam.view(n, runs)[imp, (gi[imp] // lam)]

            # --- restarts: recycle converged slots in place
            stop, _ = es.stopped()                      # reasons accumulate inside es
            k_stop = int(stop.sum())
            if k_stop:
                m_new = draw_uniform(k_stop, gen)
                src = torch.full((k_stop,), 3, dtype=torch.long, device=DEVICE)
                n_el = int(round(k_stop * cfg["p_elite"])) if have_elite else 0
                if n_el:                                         # reseed from other instances
                    ok = top_x[:, 0]
                    pick = torch.randint(len(ok), (n_el,), device=DEVICE, generator=gen)
                    m_new[:n_el] = ok[pick] + 0.10 * torch.randn(n_el, D_ANG, device=DEVICE, generator=gen)
                    src[:n_el] = 4
                n_restart += es.restart(stop, m_new)
                meta["total_runs"] += k_stop
                fam[stop] = src
                meta["restart_src"]["restart-elite"] += n_el
                meta["restart_src"]["restart-rand"] += k_stop - n_el

            # --- history
            dd = es.D.max(1).values / es.D.min(1).values.clamp_min(1e-20)
            for k_, v_ in (("wave", w), ("gen", gstep), ("evals", ev_cum),
                           ("best_mean", top_p[:, 0].mean().item()),
                           ("best_med", top_p[:, 0].median().item()),
                           ("pop_mean", p.mean().item()),
                           ("sigma_med", es.sigma.median().item()),
                           ("sigma_lo", torch.quantile(es.sigma, 0.10).item()),
                           ("sigma_hi", torch.quantile(es.sigma, 0.90).item()),
                           ("cond_med", dd.median().item()),
                           ("restarts", n_restart), ("t", time.time() - t0)):
                hist[k_].append(v_)
            gstep += 1
            if log and g % max(1, gens // 5) == 0:
                print(f"  wave {w} lam={lam:3d}  gen {g:4d}/{gens}  best P = {top_p[:, 0].mean():.5f}"
                      f"  sigma~{es.sigma.median():.3f}  restarts={n_restart}"
                      f"  ({hist['t'][-1]:.0f}s)", flush=True)
        for rn, rv in es.stop_counts.items():          # accumulated GPU-side, read once per wave
            meta["stop_counts"][rn] += rv
        meta["waves"].append(dict(wave=w, lam=lam, gens=gens, best=top_p[:, 0].mean().item()))

    meta.update(best_fam=best_fam.cpu().numpy(), restarts=n_restart, evals_per_instance=ev_cum,
                cma_time=time.time() - t0)
    return top_x, top_p, {k: np.array(v) for k, v in hist.items()}, meta


def cma_solve(h_np, cfg, seed=0, log=True):
    """Search, then hand the top-`keep` to Adam. -> (gamma, beta, p, hist, meta)."""
    top_x, top_p, hist, meta = cma_search(h_np, cfg, seed, log)
    h = torch.as_tensor(h_np, dtype=torch.float32, device=DEVICE)
    t0 = time.time()
    px, pp = adam_refine(h, top_x, cfg["polish"], cfg["polish_lr"], cfg["adam_rows"], 1)
    meta["polish_time"] = time.time() - t0
    meta["p_cma_only"] = top_p[:, 0].mean().item()
    meta["p_polished"] = pp[:, 0].mean().item()
    best = px[:, 0].cpu().numpy()
    return best[:, :P_DEPTH], best[:, P_DEPTH:], pp[:, 0].cpu().numpy(), hist, meta


def score(h_np, gamma, beta, chunk=2048):
    """Re-score with the organisers' simulator, chunked."""
    out = []
    for s in range(0, len(h_np), chunk):
        t = lambda a: torch.as_tensor(a[s:s + chunk], dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            out.append(qaoa.p_ground(t(h_np), t(gamma), t(beta)).cpu().numpy())
    return np.concatenate(out)

## 6. Watch one run

A short run on a slice of `h_train`, plotted, before committing the full budget. What to look for:

- **best-so-far** should climb in steps, not smoothly — each step is a restart landing in a better
  basin. A smooth curve means the restarts are not doing anything and the budget belongs elsewhere.
- **population mean** sits far below best-so-far and *drops* when slots restart. The gap between
  the two lines is how much exploring the search is still doing.
- **sigma** is the diagnostic that matters. It should fall by orders of magnitude within a run
  (converging) and jump back to `sigma0` on restart. If the median never falls, `sigma0` is too
  large or `lam` is too small; if it collapses immediately everywhere, the restarts are firing too
  late and evaluations are being burned on converged runs.
- **wave boundary** (dashed): the population doubles. Expect a step down in population mean and a
  slower, steadier climb in best-so-far.

In [ ]:
N_PROBE = 6 if QUICK else 32
PCFG = dict(CFG, cma_evals=CFG["cma_evals"] // (1 if QUICK else 3))
hp = h_train[:N_PROBE]

_, _, p_probe, hp_hist, hp_meta = cma_solve(hp, PCFG, seed=1)
print(f"\nprobe: {N_PROBE} instances, {hp_meta['evals_per_instance']:,} evals/instance")
print(f"  CMA-ES alone   : {hp_meta['p_cma_only']:.5f}")
print(f"  + Adam polish  : {hp_meta['p_polished']:.5f}"
      f"   (+{(hp_meta['p_polished']/max(hp_meta['p_cma_only'],1e-9)-1)*100:.1f}%)")
print(f"  restarts       : {hp_meta['restarts']}  over {hp_meta['total_runs']} CMA-ES runs total")
print(f"  wall clock     : {hp_meta['cma_time']:.0f}s search + {hp_meta['polish_time']:.0f}s polish")

H = hp_hist
wb = [H["gen"][H["wave"] == w][0] for w in np.unique(H["wave"])][1:]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))

ax[0].plot(H["evals"], H["best_mean"], label="best so far (mean)")
ax[0].plot(H["evals"], H["pop_mean"], lw=0.8, alpha=0.7, label="population mean")
ax[0].set_xlabel("forward evals per instance"); ax[0].set_ylabel("P(ground)")
ax[0].set_title("convergence"); ax[0].legend(fontsize=8)

ax[1].fill_between(H["gen"], H["sigma_lo"], H["sigma_hi"], alpha=0.25, label="10-90%")
ax[1].plot(H["gen"], H["sigma_med"], label="median")
ax[1].set_yscale("log"); ax[1].set_xlabel("generation"); ax[1].set_ylabel("sigma")
ax[1].set_title("step size across runs"); ax[1].legend(fontsize=8)

ax[2].plot(H["gen"], H["restarts"])
ax[2].set_xlabel("generation"); ax[2].set_ylabel("cumulative restarts")
ax[2].set_title(f"restarts ({hp_meta['restarts']} total)")

for a in ax:
    for b in wb:
        a.axvline(H["evals"][H["gen"] == b][0] if a is ax[0] else b, ls="--", c="k", lw=0.8, alpha=0.5)
plt.tight_layout(); plt.show()

## 7. Head-to-head at a matched budget

The claim to test is not "CMA-ES is good" but "CMA-ES spends a fixed evaluation budget better than
the alternatives". Four arms, all given the same forward-eval-equivalents per instance and the same
instances, differing only in *how* the budget is spent:

| arm | budget split |
|---|---|
| **A — Adam multistart** | all of it: `k` uniform random starts, gradient descent from each (notebook 02) |
| **B — Sobol funnel** | half on a cheap screen, half on Adam from the survivors (notebook 03, scaled down) |
| **C — CMA-ES alone** | all of it inside the evolution strategy, Hansen's restart triggers |
| **D — CMA-ES + polish** | 75% CMA-ES, 25% Adam on its top candidates |
| **E — CMA-ES, eager restarts** | as C, but triggers tightened to recycle runs roughly every 25 generations |

Three separate things get decided here. **C vs A** is the derivative-free handicap from the header:
whether the budget is past the crossover on this hardware. **D vs C** is whether the hybrid split
is worth it, or whether CMA-ES already reaches the bottom of its basin unaided. **E vs C** is the
restart-aggressiveness knob — the one setting this notebook is named after, and the one the probe
says is counter-intuitive.

If A or B wins outright, the honest conclusion is that this notebook is the wrong tool here and
notebook 03 keeps the budget.

In [ ]:
B_EVAL = 2000 if QUICK else 24_000        # forward-eval-equivalents per instance, every arm
hh = torch.as_tensor(h_train[:N_PROBE], dtype=torch.float32, device=DEVICE)
gen0 = lambda s=7: torch.Generator(device=DEVICE).manual_seed(s)
arms = {}

# --- A: uniform random starts + Adam (notebook 02) ---------------------------
kA = 32 if not QUICK else 8
stepsA = max(1, B_EVAL // 3 // kA)
t0 = time.time()
_, pA = adam_refine(hh, draw_uniform(N_PROBE * kA, gen0()).view(N_PROBE, kA, D_ANG),
                    stepsA, 0.06, CFG["adam_rows"], 1)
arms["A  adam multistart"] = (pA[:, 0].mean().item(), time.time() - t0, f"{kA} starts x {stepsA} steps")

# --- B: Sobol screen + Adam (notebook 03, budget-matched) --------------------
n_scr = B_EVAL // 2
kB = 32 if not QUICK else 8
stepsB = max(1, (B_EVAL - n_scr) // 3 // kB)
t0 = time.time()
candB = draw_screened(hh, kB, n_scr, gen0(), 7, CFG["eval_rows"])
_, pB = adam_refine(hh, candB, stepsB, 0.06, CFG["adam_rows"], 1)
arms["B  sobol funnel"] = (pB[:, 0].mean().item(), time.time() - t0,
                           f"screen {n_scr} -> {kB} x {stepsB} steps")

# --- C / D: CMA-ES, all budget vs 75/25 split --------------------------------
cfgC = dict(CFG, cma_evals=B_EVAL, keep=CFG["keep"], polish=0)
t0 = time.time()
xC, pC, _, _ = cma_search(h_train[:N_PROBE], cfgC, seed=7, log=False)
arms["C  cma-es alone"] = (pC[:, 0].mean().item(), time.time() - t0,
                           f"{cfgC['runs']}x{cfgC['lam']} runs, no gradient")

cfgD = dict(CFG, cma_evals=int(B_EVAL * 0.75))
stepsD = max(1, int(B_EVAL * 0.25) // 3 // CFG["keep"])
t0 = time.time()
xD, pD, _, _ = cma_search(h_train[:N_PROBE], cfgD, seed=7, log=False)
_, pDp = adam_refine(hh, xD, stepsD, CFG["polish_lr"], CFG["adam_rows"], 1)
arms["D  cma-es + polish"] = (torch.maximum(pDp[:, 0], pD[:, 0]).mean().item(), time.time() - t0,
                              f"75% cma, then {CFG['keep']} x {stepsD} adam steps")

# --- E: same as C, but recycling runs eagerly --------------------------------
cfgE = dict(cfgC, tolfun=1e-7, tolx=1e-8, stag_gens=25, hist_len=15)
t0 = time.time()
_, pE, _, mE = cma_search(h_train[:N_PROBE], cfgE, seed=7, log=False)
arms["E  cma-es eager restart"] = (pE[:, 0].mean().item(), time.time() - t0,
                                   f"{mE['restarts']} restarts vs C's few")

base = arms["A  adam multistart"][0]
print(f"budget = {B_EVAL:,} eval-equivalents per instance, n = {N_PROBE}\n")
print(f"{'arm':22s} {'mean P':>8s} {'vs A':>7s} {'s':>6s}   split")
for k, (v, t, note) in sorted(arms.items(), key=lambda kv: -kv[1][0]):
    print(f"{k:22s} {v:8.5f} {v/base:6.2f}x {t:6.0f}   {note}")

winner = max(arms.items(), key=lambda kv: kv[1][0])[0]
print(f"\nbest at this budget: {winner}")
if not winner.startswith(("C", "D", "E")):
    print("CMA-ES does not pay for itself at this budget -- notebook 03 keeps it.")
cc, ee = arms["C  cma-es alone"][0], arms["E  cma-es eager restart"][0]
print(f"restart aggressiveness: eager is {ee/cc:.2f}x conservative "
      f"({'eager wins -- lower stag_gens/tolfun in CFG' if ee > cc else 'keep Hansen defaults'})")

plt.figure(figsize=(6, 3))
ks = list(arms)
plt.barh(ks, [arms[k][0] for k in ks], color=["#888" if k[0] in "AB" else "#2a7" for k in ks])
plt.axvline(base, ls="--", c="k", lw=0.8)
plt.xlabel("mean P(ground)"); plt.title(f"matched budget: {B_EVAL:,} evals/instance")
plt.tight_layout(); plt.show()

## 8. Full run on `h_train`

The number the main-stage leaderboard reports, directly comparable with notebooks 02 and 03.

In [ ]:
t0 = time.time()
gamma, beta, p_fit, hist, meta = cma_solve(h_train, CFG, seed=SEED)
elapsed = time.time() - t0

p = score(h_train, gamma, beta)          # re-scored with the organisers' simulator
assert np.abs(p - p_fit).max() < 1e-4, "optimiser and scorer disagree"

print(f"\n{'='*60}")
print(f"  CMA-ES + RESTARTS  (n={len(h_train)} h_train instances)")
print(f"{'='*60}")
for w in meta["waves"]:
    print(f"  after wave {w['wave']} (lam={w['lam']:3d}, {w['gens']:4d} gens) : {w['best']:.5f}")
print(f"  after Adam polish            : {meta['p_polished']:.5f}")
print(f"  {'-'*56}")
print(f"  mean   P(ground) : {p.mean():.5f}   <- the score")
print(f"  median P(ground) : {np.median(p):.5f}")
print(f"  min / max        : {p.min():.5f} / {p.max():.5f}")
print(f"  random-angle floor: {1/2**N_QUBITS:.5f}")
print(f"  wall clock       : {elapsed:.0f}s  ({elapsed/len(h_train)*1000:.0f} ms/instance)")
print(f"{'='*60}")

np.savez_compressed("cma_angles.npz", h=h_train, gamma=gamma, beta=beta, p_ground=p)
with open("cma_history.csv", "w", newline="") as f:
    wcsv = csv.writer(f); wcsv.writerow(hist.keys())
    wcsv.writerows(zip(*hist.values()))
with open("cma_meta.json", "w") as f:
    json.dump({k: (v.tolist() if isinstance(v, np.ndarray) else v)
               for k, v in meta.items()} | dict(cfg=CFG, score=float(p.mean())), f, indent=2)
print("wrote cma_angles.npz, cma_history.csv, cma_meta.json")

### What the run did

Four things worth reading off a finished search, in the order they change what you would do next:

1. **restart triggers** — which criterion is ending runs. `stagnation` dominating means runs are
   being cut off while still improving (raise `stag_gens`); `tolfun`/`tolx` dominating means they
   are genuinely converging and the budget could go to more runs instead of longer ones.
2. **winning seed family** — whether `uniform`, `ramp`, `screened` or a restart produced each
   instance's best angles. This is the direct evidence for how to set `CFG['init']` and `p_elite`.
3. **per-wave gain** — what doubling `lambda` actually bought. A flat second wave means `waves=1`
   and a bigger `cma_evals` is the better spend.
4. **score spread** — a long left tail means a few instances are stuck in bad basins, which is a
   restart problem, not a polish problem.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.4))

# 1. restart triggers
sc = {k: v for k, v in meta["stop_counts"].items() if v}
ax[0].bar(range(len(sc)), list(sc.values()), color="#c66")
ax[0].set_xticks(range(len(sc))); ax[0].set_xticklabels(list(sc), rotation=30, ha="right", fontsize=8)
ax[0].set_ylabel("runs ended"); ax[0].set_title(f"restart trigger ({meta['restarts']} total)")

# 2. which seeding family produced each instance's winner
bf = meta["best_fam"]
cnt = [int((bf == i).sum()) for i in range(len(FAMILIES))]
ax[1].bar(range(len(FAMILIES)), cnt, color="#57a")
ax[1].set_xticks(range(len(FAMILIES)))
ax[1].set_xticklabels(FAMILIES, rotation=30, ha="right", fontsize=8)
ax[1].set_ylabel("instances won"); ax[1].set_title("winning seed family")

# 3. convergence with wave boundaries
ax[2].plot(hist["evals"], hist["best_mean"], label="best so far")
ax[2].plot(hist["evals"], hist["pop_mean"], lw=0.8, alpha=0.6, label="population")
for w in np.unique(hist["wave"])[1:]:
    ax[2].axvline(hist["evals"][hist["wave"] == w][0], ls="--", c="k", lw=0.8, alpha=0.6)
ax[2].axhline(meta["p_polished"], ls=":", c="g", lw=1, label="after polish")
ax[2].set_xlabel("forward evals per instance"); ax[2].set_ylabel("P(ground)")
ax[2].set_title("convergence"); ax[2].legend(fontsize=8)

# 4. score distribution
ax[3].hist(p, bins=50, color="#7a7")
ax[3].axvline(p.mean(), c="k", ls="--", label=f"mean {p.mean():.4f}")
ax[3].set_xlabel("P(ground)"); ax[3].set_ylabel("instances")
ax[3].set_title("per-instance score"); ax[3].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'seed family':16s} {'won':>5s}   share")
for i, f in enumerate(FAMILIES):
    print(f"  {f:14s} {cnt[i]:5d}   {cnt[i]/len(bf)*100:5.1f}%")
print(f"\ndistinct CMA-ES runs executed: {meta['total_runs']:,} "
      f"({meta['total_runs']/len(h_train):.0f} per instance)")
print(f"restart reseeding: " + ", ".join(f"{k} {v}" for k, v in meta["restart_src"].items()))

## 9. Inference-budget check

The rules give 10 minutes for all 500 `h_test` instances. This search *is* the inference, so the
wall clock above is the cost that counts.

In [ ]:
proj = elapsed / len(h_train) * 500
print(f"measured        : {elapsed/len(h_train)*1000:.1f} ms / instance")
print(f"projected (500) : {proj:.0f}s of a 600s budget  ({proj/600*100:.0f}% used)")

if proj > 600:
    head = 600 / proj
    print(f"\nOVER BUDGET by {proj/600:.1f}x. Cheapest quality loss first:")
    print(f"  - cma_evals {CFG['cma_evals']:,} -> {max(2000, int(CFG['cma_evals']*head)):,} "
          f"(CMA is {WORK['cma']/sum(WORK.values())*100:.0f}% of the work)")
    print(f"  - waves {CFG['waves']} -> 1  (§8 panel 3 shows what the second wave bought)")
    print(f"  - keep {CFG['keep']} -> {max(4, int(CFG['keep']*head))}, polish stays")
    print("\nOr use this notebook offline as a label/candidate generator (§11).")
else:
    print(f"\nWithin budget with {600-proj:.0f}s to spare. Raise cma_evals to use it.")

## 10. Submission

Uses `h_test.npy` as soon as it is present, otherwise `h_train.npy` so the cell stays runnable.
Angles are searched for whichever file is loaded — nothing is reused from the run above.

In [ ]:
def write_submission(path, gamma, beta):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id"] + [f"gamma_{i}" for i in range(P_DEPTH)]
                          + [f"beta_{i}" for i in range(P_DEPTH)])
        for i, (gi, bi) in enumerate(zip(gamma, beta)):
            w.writerow([i] + [f"{x:.8f}" for x in gi] + [f"{x:.8f}" for x in bi])


try:
    h_sub, src = np.load(find_file("h_test.npy")).astype(np.float64), "h_test.npy"
    t0 = time.time()
    g_sub, b_sub, _, _, _ = cma_solve(h_sub, CFG, seed=SEED)
    infer_time = time.time() - t0
except FileNotFoundError:
    h_sub, src = h_train, "h_train.npy (h_test not released yet)"
    g_sub, b_sub, infer_time = gamma, beta, elapsed

write_submission("submission.csv", g_sub, b_sub)
p_sub = score(h_sub, g_sub, b_sub)
print(f"wrote submission.csv from {src}")
print(f"  rows           : {len(h_sub)}")
print(f"  mean P(ground) : {p_sub.mean():.5f}")
print(f"  inference time : {infer_time:.0f}s of 600s")
print(f"  identical rows : {len(np.unique(np.round(np.c_[g_sub, b_sub], 6), axis=0))} distinct "
      f"of {len(h_sub)}  (constant submissions score 0)")

## 11. What this is for

**A measurement, first.** §7 is the point of the notebook: it prices CMA-ES against the two search
baselines at an identical budget, on the same instances. The header predicted that a
derivative-free method would be handicapped where the gradient is free, and §7 either confirms
that or refutes it. Either answer is worth having — an unmeasured method in the writeup is worth
less than a measured one that lost.

**A different kind of candidate.** `cma_angles.npz` holds angles found by a search that explores
along a *learned covariance* rather than along the gradient, so its winners tend to sit in
different basins from notebook 03's. Two uses:

- merge them into notebook 03's elite pool and the transformer's rollout starts — a union of two
  searches is strictly better supervision than either;
- take the per-instance max over both notebooks as the evaluation ceiling, which is a tighter
  oracle than either alone.

**What it is not** is a solution to the stated task. The rules ask for a *model* that maps `h` to
angles in under 10 minutes for 500 instances; this is a search that reads `h` and optimises. It is
a reference score, a label generator, and — via §8's seed-family breakdown — evidence about where
good angles actually live, which is what the model has to learn.